### Today we'll be using the dataset from NOAA to visualize temperature data from stations around the world using plotly.

First, let's import the necessary packages.

In [3]:
import pandas as pd
import numpy as np
import sqlite3

In [4]:
import plotly
import plotly.io as pio
pio.renderers.default="iframe"
from plotly import express as px

# Making the Database

In earlier lectures, we created a database called `temps.db` that contains two tables: `stations` and `temperatures`. If you have this database, make sure it is saved in this directory and then skip ahead to "adding the `countries` table". If not, you can run the following code to make it.

### Creating and populating the Database

In [8]:
import os
# create folder named "datafiles" if it does not exist
if not os.path.exists("datafiles"): 
    os.mkdir("datafiles")

# download the files
import urllib.request
intervals = [f"{i}-{i+9}" for i in range(1901, 2020, 10)]
for interval in intervals:
    url = \
    f"https://raw.githubusercontent.com/PIC16B-ucla/24F/main/datasets/noaa-ghcn/decades/{interval}.csv"
    urllib.request.urlretrieve(url, f"datafiles/{interval}.csv")

In [9]:
conn = sqlite3.connect("temps.db") # create a database in current directory called temps.db

In [10]:
def prepare_df(df):
    """
    prepares a piece of wide format dataframe into a long format data frame
    """
    # melt to the long format table
    df = df.melt(
        id_vars = ["ID", "Year"],
        value_vars = [f"VALUE{i}" for i in range(1, 13)],
        var_name = "Month",
        value_name = "Temp"
    )

    # cleaning month and temp
    df["Month"] = df["Month"].str[5:].astype(int)
    df["Temp"]  = df["Temp"] / 100
    df = df[~np.isnan(df["Temp"])]

    return df

#### Populate the `temperatures` table:

In [12]:
intervals = [f"{i}-{i+9}" for i in range(1901, 2020, 10)]
for i, interval in enumerate(intervals):
    filepath = f"datafiles/{interval}.csv"
    df = pd.read_csv(filepath)
    df = prepare_df(df)
    df.to_sql("temperatures", conn, if_exists = "append", index = False)

#### Populate the `stations` table:

In [14]:
filename = \
"https://raw.githubusercontent.com/PIC16B-ucla/25W/refs/heads/main/datasets/noaa-ghcn/station-metadata.csv"
stations = pd.read_csv(filename)
stations.to_sql("stations", conn, if_exists = "replace", index=False)

27585

#### Check that the database is populated correctly:

In [16]:
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
print(cursor.fetchall())

[('temperatures',), ('countries',), ('stations',)]


The output should read "[('temperatures',), ('stations',)]". Now we're ready to move on to the `countries` table! But first, let's close the connection to be safe.

In [18]:
conn.close()

### Create and populate the `countries` table:

In [20]:
with sqlite3.connect("temps.db") as conn:
    cursor = conn.cursor()
    
    filename = \
    "https://raw.githubusercontent.com/mysociety/gaze/master/data/fips-10-4-to-iso-country-codes.csv"
    countries = pd.read_csv(filename)
    countries.head()
    countries.to_sql("countries", conn, if_exists = "replace", index=False)
    cursor.execute("SELECT sql FROM sqlite_master WHERE type='table';")
    for result in cursor.fetchall():
        print(result[0])

CREATE TABLE "temperatures" (
"ID" TEXT,
  "Year" INTEGER,
  "Month" INTEGER,
  "Temp" REAL
)
CREATE TABLE "stations" (
"ID" TEXT,
  "LATITUDE" REAL,
  "LONGITUDE" REAL,
  "STNELEV" REAL,
  "NAME" TEXT
)
CREATE TABLE "countries" (
"FIPS 10-4" TEXT,
  "ISO 3166" TEXT,
  "Name" TEXT
)


That should have created a new table called `countries`. If the database was populated correctly, your output should tell you the names and columns of each table, as well as the type of data in each column. Now we're ready to take a look at this data!

# Visualizing the Data Using Plotly

## Question 1: How does the average yearly change in temperature vary within a given country?

To answer this question, we'll make a geographic scatterplot using the function `temperature_coefficient_plot`. This function uses another function, `coef`, which we'll define first. `coef` is used to calculate the first coefficient in the linear regression for `Temp` vs. `Year`, which we will use to estimate the average yearly change in temperature over our interval.

In [25]:
from sklearn.linear_model import LinearRegression

def coef(data_group):
    """
    returns the first coefficient in the linear regression for the given dataframe.
    """
    x = data_group[["Year"]] # 2 brackets because X should be a df
    y = data_group["Temp"]   # 1 bracket because y should be a series
    LR = LinearRegression()
    LR.fit(x, y)
    return round(LR.coef_[0], 4)

`temperature_coefficient_plot` also uses another function, `query_climate_database` to access the database, which is defined in the file `climate_database.py` (make sure you have this saved in this directory!).

In [27]:
from climate_database import query_climate_database
import inspect
print(inspect.getsource(query_climate_database))

def query_climate_database(db_file, country, year_begin, year_end, month):
    
    """
    makes a query to db_file for climate data for given arguments and
    returns a dataframe with relevant data    
    """
    
    cols = 'stations.NAME, LATITUDE, LONGITUDE, countries.Name AS Country, Year, Month, Temp'
    ISOcmd = \
    f"""
        SELECT {cols} 
        FROM ((temperatures 
        INNER JOIN stations ON temperatures.ID = stations.ID)
        LEFT JOIN countries ON SUBSTR(stations.ID, 1, 2) = "ISO 3166")
        WHERE countries.Name = \'{country}\' AND Year >= {year_begin} \
        AND Year <= {year_end} AND Month = {month}
    ;"""
    FIPScmd = \
    f"""
        SELECT {cols} 
        FROM ((temperatures 
        INNER JOIN stations ON temperatures.ID = stations.ID)
        LEFT JOIN countries ON SUBSTR(stations.ID, 1, 2) = "FIPS 10-4")
        WHERE countries.Name = \'{country}\' AND Year >= {year_begin} \
        AND Year <= {year_end} AND Month = {month}
    ;"""

   

In [28]:
def temperature_coefficient_plot(db_file, country, year_begin, year_end, 
                                 month, min_obs, **kwargs):
    
    """
    plots average yearly temperature change for 
    a given country over a given interval on 
    an interactive geographic scatterplot, returns the figure
    """
    
    df = query_climate_database(db_file, country, year_begin, year_end, month)

    #filter out stations with insufficient data
    
    df["years obs"] = df.groupby(["NAME"])["Year"].transform("count")
    df = df[df["years obs"] >= min_obs]
        
    #calculate yearly increase in temp and add to dataframe
    
    df = (df.groupby("NAME").apply(coef).reset_index()
          .rename(columns = {0 : "Estimated Yearly Increase"})
          .merge(df, on = "NAME")
          .drop_duplicates(subset = 'NAME', ignore_index = True))
    
    figtitle = \
    f"Yearly Increase in Temperature in Month {month} in {country}, Years {year_begin} - {year_end}"

    
    fig = px.scatter_mapbox(df,
                        lat = "LATITUDE",
                        lon = "LONGITUDE",
                        hover_name = "NAME",
                        color = "Estimated Yearly Increase",
                        title = figtitle,
                        **kwargs
                       )
                        
    # fig.update_layout(mapbox_style="carto-positron")
    # fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
    return fig

Our first plotting function is now fully defined. Let's test it out!

In [30]:
color_map = px.colors.diverging.RdGy_r # choose a colormap

fig = temperature_coefficient_plot("temps.db", "India", 1980, 2020, 1, 
                                   min_obs = 10,
                                   zoom = 2,
                                   mapbox_style="carto-positron",
                                   color_continuous_scale=color_map)

fig.show()

C:\Users\Beatrice NT\AppData\Local\Temp\ipykernel_13316\1511400234.py:19: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [31]:
color_map = 'portland' # choose a colormap

fig = temperature_coefficient_plot("temps.db", "United States", 1960, 2015, 6, 
                                   min_obs = 10,
                                   zoom = 2,
                                   mapbox_style="carto-positron",
                                   color_continuous_scale=color_map,
                                  opacity = 0.6)

fig.show()

C:\Users\Beatrice NT\AppData\Local\Temp\ipykernel_13316\1511400234.py:19: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



## Question 2: How does the average temperature at stations in a given country vary over time during winter vs. summer?

We'll define another function to visualize the data to answer this question and another query function to help it.

In [34]:
def averages_query(db_file, country, begin, end):
    
    """
    makes a query to the database for the given country during interval,
    returns a dataframe with averaged temperatures for each station
    during the months January and June
    """
    
    ISOcmd = \
    f"""
        SELECT stations.NAME, Month, Year, AVG(Temp)
        FROM ((temperatures 
        INNER JOIN stations ON temperatures.ID = stations.ID)
        LEFT JOIN countries ON SUBSTR(stations.ID, 1, 2) = "ISO 3166")
        WHERE countries.Name = \'{country}\' AND Year >= {begin} AND Year <= {end} \
        AND (Month = 1 OR Month = 6)
        GROUP BY Month
    ;"""
    FIPScmd = \
    f"""
        SELECT stations.NAME, Month, Year, AVG(Temp)
        FROM ((temperatures 
        INNER JOIN stations ON temperatures.ID = stations.ID)
        LEFT JOIN countries ON SUBSTR(stations.ID, 1, 2) = "FIPS 10-4")
        WHERE countries.Name = \'{country}\' AND Year >= {begin} AND Year <= {end} \
        AND (Month = 1 OR Month = 6)
        GROUP BY stations.NAME, Month
    ;"""

    with sqlite3.connect(db_file) as conn:
        cursor = conn.cursor()
        df = pd.read_sql_query(ISOcmd, conn)
        df2 = pd.read_sql_query(FIPScmd, conn)
        df = df.merge(df2, how = 'outer')
    return df

In [35]:
def seasonal_temps_plot(db_file, country, year_begin, year_end, 
                        interval_length = 10, width = 600, height = 350, **kwargs):
        
    """
    makes a boxplot of average temperatures over given time span 
    for stations in given country during January and June
    """    
        
        
    if interval_length == 10: interval = 'Decade'
    else: interval = f"{interval_length} - Year Intervals"
    
    #call averages to get df
    
    i = year_begin
    df = pd.DataFrame(columns = ['NAME', 'Month', 'Year', 'AVG(Temp)'])
    while i + interval_length -1 <= year_end:
        df = df.merge(averages_query(db_file, country, i, i + interval_length -1),
                      how = 'outer',
                      on = ['NAME', 'Month', 'Year', 'AVG(Temp)'])
        i += interval_length


    #make the plot
    df = df[df['Year'] % interval_length == year_begin % interval_length] 
    #to drop the rows that don't have data for the entire interval
    
    df['Month'] = df['Month'].map({1 : 'January', 6 : 'June'})
    
    fig = px.box(data_frame = df,
                 x = "Year",
                 y = "AVG(Temp)",
                 hover_name = "NAME",
                 width = width,
                 height = height,
                 color = "Month",
                 title = f"Mean Temps by {interval} for Stations in {country} from {year_begin} to {year_end}",
                 **kwargs)
    fig.update_layout(yaxis_title = 'Average Temperatures')
    fig.update_layout(margin={"r":80,"t":50,"l":40,"b":0})

    return fig

That's all we need. Let's try it out!

In [37]:
fig = seasonal_temps_plot("temps.db", "India", 1980, 2020)

fig.show()

In [38]:
fig = seasonal_temps_plot("temps.db", "United States", 1960, 2020, width = 700, height = 400)

fig.show()

## Question 3: How much does temperature range vary across latitudes in different years?

Again, let's make a query function and a plot function to answer this question.

In [41]:
def temp_range_query(db_file, year):
    
    """
    makes a query to database for the given year, returns a dataframe
    with max, min, and range of temperature, averaged across stations in each country
    """
    
    cols = 'stations.NAME, LATITUDE, countries.Name AS Country, Year, MAX(Temp), MIN(Temp)'
    ISOcmd = \
    f"""
        SELECT {cols} 
        FROM ((temperatures 
        INNER JOIN stations ON temperatures.ID = stations.ID)
        LEFT JOIN countries ON SUBSTR(stations.ID, 1, 2) = "ISO 3166")
        WHERE Year = {year}
        GROUP BY countries.Name, stations.NAME
    ;"""
    FIPScmd = \
    f"""
        SELECT {cols} 
        FROM ((temperatures 
        INNER JOIN stations ON temperatures.ID = stations.ID)
        LEFT JOIN countries ON SUBSTR(stations.ID, 1, 2) = "FIPS 10-4")
        WHERE Year = {year}
        GROUP BY countries.Name, stations.NAME
    ;"""
    
    with sqlite3.connect(db_file) as conn:
        cursor = conn.cursor()
        df = pd.read_sql_query(ISOcmd, conn)
        df2 = pd.read_sql_query(FIPScmd, conn)
        df = df.merge(df2, how = 'outer')
        
    df['Temp Range'] = df['MAX(Temp)'] - df['MIN(Temp)']
    df['Temp Range'] = df.groupby('Country')['Temp Range'].transform('mean')
    df['LATITUDE'] = df.groupby('Country')['LATITUDE'].transform('mean')

    df.drop_duplicates(subset = ['Country'], inplace = True, ignore_index = True)
    return df

In [42]:
def temp_range_compare(db_file, year1, year2, opacity = 0.5, **kwargs):
    
    """
    makes a scatterplot of temperature range vs. latitude averaged across stations
    in each country during the given years, returns the figure
    """

    df = pd.concat([temp_range_query(db_file, year1), temp_range_query(db_file, year2)])
    df.dropna(inplace = True, ignore_index = True)
    fig = px.scatter(data_frame = df,
                 x = "LATITUDE",
                 y = "Temp Range",
                 width = 700,
                 height = 400,
                 hover_name = 'Country',
                 hover_data = ['MAX(Temp)', 'MIN(Temp)'],
                 facet_col = 'Year',
                 color = 'MAX(Temp)',
                 opacity = opacity,
                 title = \
                     f"Average Temperature Range vs. Latitude by Country in {year1} and {year2}",
                 **kwargs
                )
    
    return fig

In [43]:
temp_range_compare('temps.db', year1 = 1980, year2 = 2000)

In [44]:
temp_range_compare('temps.db', 1901, 1970)

Cool! We can see that countries farther from the equator tend to have a larger temperature range, and the color tells us that colder (in terms of maximum temperature) countries tend to have a larger temperature range, and that these trends appear consistently throughout different years (although the maximum temperatures seem to have increased in general).